In [39]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
import scipy

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("yfinance :", yf.__version__)
print("Toutes les bibliothèques sont correctement chargées.")

Pandas : 3.0.5
NumPy : 2.5.2
yfinance : 1.6.0
Toutes les bibliothèques sont correctement chargées.


In [40]:
entreprises = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "BNP Paribas": "BNP.PA"
}

symboles = list(entreprises.values())

print("Symboles sélectionnés :", symboles)

Symboles sélectionnés : ['GOOGL', 'AAPL', 'BNP.PA']


In [41]:
from datetime import date

date_debut = "2023-01-01"
date_fin = date.today().strftime("%Y-%m-%d")

donnees = yf.download(
    symboles,
    start=date_debut,
    end=date_fin,
    auto_adjust=True,
    progress=False
)

print("Nombre de lignes et de colonnes :", donnees.shape)

display(donnees.head())

Nombre de lignes et de colonnes : (946, 15)


Price            Close                              High             \
Ticker            AAPL     BNP.PA      GOOGL        AAPL     BNP.PA   
Date                                                                  
2023-01-02         NaN  42.455944        NaN         NaN  42.579610   
2023-01-03  122.876740  43.290688  88.336693  128.604497  43.568936   
2023-01-04  124.144127  45.068382  87.305832  126.403797  45.300255   
2023-01-05  122.827606  45.300251  85.442360  125.529381  45.578499   
2023-01-06  127.346954  45.686710  86.572334  128.005203  45.725355   

Price                         Low                              Open  \
Ticker          GOOGL        AAPL     BNP.PA      GOOGL        AAPL   
Date                                                                  
2023-01-02        NaN         NaN  41.543910        NaN         NaN   
2023-01-03  90.249730  121.992521  42.409571  87.741960  127.995375   
2023-01-04  89.853243  122.886574  43.568935  86.502946  124.664832   
2023-01-05  86.800321  122.572171  44.774672  85.145001  124.900605   
2023-01-06  86.919264  122.699905  45.137945  84.114136  123.800267   

Price                                  Volume                         
Ticker         BNP.PA      GOOGL         AAPL     BNP.PA       GOOGL  
Date                                                                  
2023-01-02  41.667576        NaN          NaN  1815541.0         NaN  
2023-01-03  42.494591  88.802555  112117500.0  2816969.0  28131200.0  
2023-01-04  43.568935  89.555877   89113600.0  4045737.0  34854800.0  
2023-01-05  44.906065  86.701202   80962700.0  2872672.0  27194400.0  
2023-01-06  45.192048  86.027173   87754700.0  2907109.0  41381500.0

In [4]:
cours = donnees["Close"].copy()

# Remplacer les symboles par les noms des entreprises
noms_entreprises = {
    symbole: nom
    for nom, symbole in entreprises.items()
}

cours = cours.rename(columns=noms_entreprises)

display(cours.head(10))

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-02,NaN,42.455944,NaN
2023-01-03,122.876740,43.290688,88.336700
2023-01-04,124.144119,45.068382,87.305832
2023-01-05,122.827606,45.300255,85.442368
2023-01-06,127.346939,45.686710,86.572334
2023-01-09,127.867638,45.617149,87.246353
2023-01-10,128.437485,45.099297,87.642853
2023-01-11,131.149078,45.292530,90.715607
2023-01-12,131.070480,46.127266,90.329033


In [5]:
# Nombre de valeurs manquantes avant le traitement
print("Avant le traitement :")
display(cours.isna().sum())

# Reporter le dernier cours connu
cours_complets = cours.ffill()

# Supprimer uniquement les premières lignes encore incomplètes
cours_complets = cours_complets.dropna()

print("Après le traitement :")
display(cours_complets.isna().sum())

display(cours_complets.head())

Avant le traitement :


Ticker
Apple          28
BNP Paribas    11
Google         28
dtype: int64

Après le traitement :


Ticker
Apple          0
BNP Paribas    0
Google         0
dtype: int64

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-03,122.876740,43.290688,88.336700
2023-01-04,124.144119,45.068382,87.305832
2023-01-05,122.827606,45.300255,85.442368
2023-01-06,127.346939,45.686710,86.572334
2023-01-09,127.867638,45.617149,87.246353


In [6]:
rendements = (
    cours_complets / cours_complets.shift(1)
) - 1

# La première ligne n’a pas de journée précédente
rendements = rendements.dropna()

display(
    rendements.head(10).style.format("{:.2%}")
)

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-04 00:00:00,1.03%,4.11%,-1.17%
2023-01-05 00:00:00,-1.06%,0.51%,-2.13%
2023-01-06 00:00:00,3.68%,0.85%,1.32%
2023-01-09 00:00:00,0.41%,-0.15%,0.78%
2023-01-10 00:00:00,0.45%,-1.14%,0.45%
2023-01-11 00:00:00,2.11%,0.43%,3.51%
2023-01-12 00:00:00,-0.06%,1.84%,-0.43%
2023-01-13 00:00:00,1.01%,0.30%,1.09%
2023-01-16 00:00:00,0.00%,-0.50%,0.00%


In [7]:
performance_base100 = (
    cours_complets / cours_complets.iloc[0]
) * 100

display(performance_base100.head())

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-03,100.000000,100.000000,100.000000
2023-01-04,101.031424,104.106414,98.833024
2023-01-05,99.960014,104.642031,96.723522
2023-01-06,103.637954,105.534730,98.002680
2023-01-09,104.061711,105.374047,98.765692


In [11]:
# Performance totale sur toute la période
performance_totale = (
    cours_complets.iloc[-1] / cours_complets.iloc[0]
) - 1

# Rendement annuel composé
rendement_annuel = (
    cours_complets.iloc[-1] / cours_complets.iloc[0]
) ** (252 / len(rendements)) - 1

# Volatilité annuelle
volatilite_annuelle = (
    rendements.std() * np.sqrt(252)
)

resume = pd.DataFrame({
    "Performance totale": performance_totale,
    "Rendement annuel": rendement_annuel,
    "Volatilité annuelle": volatilite_annuelle
})

display(
    resume.style.format({
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}"
    })
)

,Performance totale,Rendement annuel,Volatilité annuelle
Ticker,,,
Apple,151.76%,28.15%,25.56%
BNP Paribas,148.32%,27.68%,26.27%
Google,290.35%,44.18%,30.41%


In [12]:
taux_sans_risque = 0.02

ratio_sharpe = (
    rendement_annuel - taux_sans_risque
) / volatilite_annuelle

resume["Ratio de Sharpe"] = ratio_sharpe

display(
    resume.style.format({
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}",
        "Ratio de Sharpe": "{:.2f}"
    })
)

,Performance totale,Rendement annuel,Volatilité annuelle,Ratio de Sharpe
Ticker,,,,
Apple,151.76%,28.15%,25.56%,1.02
BNP Paribas,148.32%,27.68%,26.27%,0.98
Google,290.35%,44.18%,30.41%,1.39


In [13]:
# Évolution d’un investissement initial de 1 €
valeur_cumulee = (1 + rendements).cumprod()

# Plus haut niveau atteint jusqu’à chaque date
sommets_historiques = valeur_cumulee.cummax()

# Écart entre la valeur actuelle et son précédent sommet
drawdowns = (
    valeur_cumulee / sommets_historiques
) - 1

# Plus forte baisse de chaque entreprise
maximum_drawdown = drawdowns.min()

resume["Maximum Drawdown"] = maximum_drawdown

display(
    resume.style.format({
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}",
        "Ratio de Sharpe": "{:.2f}",
        "Maximum Drawdown": "{:.2%}"
    })
)

,Performance totale,Rendement annuel,Volatilité annuelle,Ratio de Sharpe,Maximum Drawdown
Ticker,,,,,
Apple,151.76%,28.15%,25.56%,1.02,-33.36%
BNP Paribas,148.32%,27.68%,26.27%,0.98,-23.70%
Google,290.35%,44.18%,30.41%,1.39,-29.81%


In [14]:
niveau_confiance = 0.95

# Quantile des 5 % plus mauvais rendements quotidiens
var_95 = rendements.quantile(
    1 - niveau_confiance
)

resume["VaR historique 95 %"] = var_95

display(
    resume.style.format({
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}",
        "Ratio de Sharpe": "{:.2f}",
        "Maximum Drawdown": "{:.2%}",
        "VaR historique 95 %": "{:.2%}"
    })
)

,Performance totale,Rendement annuel,Volatilité annuelle,Ratio de Sharpe,Maximum Drawdown,VaR historique 95 %
Ticker,,,,,,
Apple,151.76%,28.15%,25.56%,1.02,-33.36%,-2.32%
BNP Paribas,148.32%,27.68%,26.27%,0.98,-23.70%,-2.62%
Google,290.35%,44.18%,30.41%,1.39,-29.81%,-2.61%


In [15]:
correlations = rendements.corr()

display(
    correlations.style.format("{:.2f}")
)

Ticker,Apple,BNP Paribas,Google
Ticker,,,
Apple,1.00,0.09,0.39
BNP Paribas,0.09,1.00,0.10
Google,0.39,0.10,1.00


In [16]:
fig_correlation = px.imshow(
    correlations,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Corrélations entre les rendements quotidiens"
)

fig_correlation.update_layout(
    template="plotly_white"
)

fig_correlation.show(renderer="browser")

In [17]:
taux_change = yf.download(
    "EURUSD=X",
    start=date_debut,
    end=date_fin,
    auto_adjust=True,
    progress=False
)

eurusd = taux_change["Close"].squeeze()
eurusd.name = "EUR/USD"

display(eurusd.head())

Date
2023-01-02    1.070973
2023-01-03    1.067771
2023-01-04    1.054685
2023-01-05    1.060637
2023-01-06    1.052222
Name: EUR/USD, dtype: float64

In [18]:
eurusd_aligne = (
    eurusd
    .reindex(cours_complets.index)
    .ffill()
    .bfill()
)

print("Valeurs manquantes :", eurusd_aligne.isna().sum())
display(eurusd_aligne.head())

Valeurs manquantes : 0


Date
2023-01-03    1.067771
2023-01-04    1.054685
2023-01-05    1.060637
2023-01-06    1.052222
2023-01-09    1.065632
Name: EUR/USD, dtype: float64

In [19]:
cours_euros = cours_complets.copy()

# Les cours Apple et Google sont initialement en dollars
cours_euros["Apple"] = (
    cours_complets["Apple"] / eurusd_aligne
)

cours_euros["Google"] = (
    cours_complets["Google"] / eurusd_aligne
)

# BNP Paribas est déjà cotée en euros
display(cours_euros.head())

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-03,115.077755,43.290688,82.729972
2023-01-04,117.707243,45.068382,82.779022
2023-01-05,115.805560,45.300255,80.557633
2023-01-06,121.026709,45.686710,82.275749
2023-01-09,119.992264,45.617149,81.872846


In [20]:
rendements_euros = (
    cours_euros / cours_euros.shift(1)
) - 1

rendements_euros = rendements_euros.dropna()

display(
    rendements_euros.head().style.format("{:.2%}")
)

Ticker,Apple,BNP Paribas,Google
Date,,,
2023-01-04 00:00:00,2.28%,4.11%,0.06%
2023-01-05 00:00:00,-1.62%,0.51%,-2.68%
2023-01-06 00:00:00,4.51%,0.85%,2.13%
2023-01-09 00:00:00,-0.85%,-0.15%,-0.49%
2023-01-10 00:00:00,-0.27%,-1.14%,-0.26%


In [21]:
montant_initial = 10_000

poids = pd.Series({
    "Apple": 0.40,
    "BNP Paribas": 0.30,
    "Google": 0.30
})

print("Somme des poids :", poids.sum())
display(poids)

Somme des poids : 1.0


Apple          0.4
BNP Paribas    0.3
Google         0.3
dtype: float64

In [22]:
rendement_portefeuille = rendements_euros.dot(poids)

display(
    rendement_portefeuille.head(10).to_frame(
        name="Rendement du portefeuille"
    ).style.format("{:.2%}")
)

,Rendement du portefeuille
Date,
2023-01-04 00:00:00,2.16%
2023-01-05 00:00:00,-1.30%
2023-01-06 00:00:00,2.70%
2023-01-09 00:00:00,-0.53%
2023-01-10 00:00:00,-0.53%
2023-01-11 00:00:00,1.99%
2023-01-12 00:00:00,0.23%
2023-01-13 00:00:00,0.21%
2023-01-16 00:00:00,0.08%


In [23]:
valeur_portefeuille = (
    montant_initial
    * (1 + rendement_portefeuille).cumprod()
)

display(
    valeur_portefeuille.head().to_frame(
        name="Valeur du portefeuille"
    ).style.format("{:,.2f} €")
)

,Valeur du portefeuille
Date,
2023-01-04 00:00:00,"10,216.37 €"
2023-01-05 00:00:00,"10,083.87 €"
2023-01-06 00:00:00,"10,356.05 €"
2023-01-09 00:00:00,"10,300.70 €"
2023-01-10 00:00:00,"10,246.16 €"


In [24]:
fig_portefeuille = px.line(
    x=valeur_portefeuille.index,
    y=valeur_portefeuille.values,
    title="Évolution de la valeur du portefeuille",
    labels={
        "x": "Date",
        "y": "Valeur du portefeuille en euros"
    }
)

fig_portefeuille.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig_portefeuille.show(renderer="browser")

In [25]:
# Valeur finale
valeur_finale = valeur_portefeuille.iloc[-1]

# Performance totale
performance_portefeuille = (
    valeur_finale / montant_initial
) - 1

# Rendement annuel composé
rendement_annuel_portefeuille = (
    valeur_finale / montant_initial
) ** (252 / len(rendement_portefeuille)) - 1

# Volatilité annuelle
volatilite_portefeuille = (
    rendement_portefeuille.std() * np.sqrt(252)
)

# Ratio de Sharpe
sharpe_portefeuille = (
    rendement_annuel_portefeuille - taux_sans_risque
) / volatilite_portefeuille

# Drawdown
valeur_cumulee_portefeuille = (
    1 + rendement_portefeuille
).cumprod()

sommet_portefeuille = (
    valeur_cumulee_portefeuille.cummax()
)

drawdown_portefeuille = (
    valeur_cumulee_portefeuille
    / sommet_portefeuille
) - 1

maximum_drawdown_portefeuille = (
    drawdown_portefeuille.min()
)

# Value at Risk historique à 95 %
var_portefeuille = (
    rendement_portefeuille.quantile(0.05)
)

In [26]:
# Valeur finale
valeur_finale = valeur_portefeuille.iloc[-1]

# Performance totale
performance_portefeuille = (
    valeur_finale / montant_initial
) - 1

# Rendement annuel composé
rendement_annuel_portefeuille = (
    valeur_finale / montant_initial
) ** (252 / len(rendement_portefeuille)) - 1

# Volatilité annuelle
volatilite_portefeuille = (
    rendement_portefeuille.std() * np.sqrt(252)
)

# Ratio de Sharpe
sharpe_portefeuille = (
    rendement_annuel_portefeuille - taux_sans_risque
) / volatilite_portefeuille

# Drawdown
valeur_cumulee_portefeuille = (
    1 + rendement_portefeuille
).cumprod()

sommet_portefeuille = (
    valeur_cumulee_portefeuille.cummax()
)

drawdown_portefeuille = (
    valeur_cumulee_portefeuille
    / sommet_portefeuille
) - 1

maximum_drawdown_portefeuille = (
    drawdown_portefeuille.min()
)

# Value at Risk historique à 95 %
var_portefeuille = (
    rendement_portefeuille.quantile(0.05)
)

In [27]:
resume_portefeuille = pd.DataFrame({
    "Montant initial": [montant_initial],
    "Valeur finale": [valeur_finale],
    "Performance totale": [performance_portefeuille],
    "Rendement annuel": [rendement_annuel_portefeuille],
    "Volatilité annuelle": [volatilite_portefeuille],
    "Ratio de Sharpe": [sharpe_portefeuille],
    "Maximum Drawdown": [maximum_drawdown_portefeuille],
    "VaR historique 95 %": [var_portefeuille]
}, index=["Portefeuille"])

display(
    resume_portefeuille.style.format({
        "Montant initial": "{:,.2f} €",
        "Valeur finale": "{:,.2f} €",
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}",
        "Ratio de Sharpe": "{:.2f}",
        "Maximum Drawdown": "{:.2%}",
        "VaR historique 95 %": "{:.2%}"
    })
)

,Montant initial,Valeur finale,Performance totale,Rendement annuel,Volatilité annuelle,Ratio de Sharpe,Maximum Drawdown,VaR historique 95 %
Portefeuille,"10,000.00 €","28,976.12 €",189.76%,33.09%,19.12%,1.63,-23.19%,-1.74%


In [28]:
montants_investis = poids * montant_initial

# Nombre théorique d’actions achetées au départ
quantites = (
    montants_investis / cours_euros.iloc[0]
)

display(
    pd.DataFrame({
        "Poids initial": poids,
        "Montant investi": montants_investis,
        "Cours initial en euros": cours_euros.iloc[0],
        "Quantité achetée": quantites
    }).style.format({
        "Poids initial": "{:.0%}",
        "Montant investi": "{:,.2f} €",
        "Cours initial en euros": "{:,.2f} €",
        "Quantité achetée": "{:.4f}"
    })
)

,Poids initial,Montant investi,Cours initial en euros,Quantité achetée
Apple,40%,"4,000.00 €",115.08 €,34.7591
BNP Paribas,30%,"3,000.00 €",43.29 €,69.2990
Google,30%,"3,000.00 €",82.73 €,36.2626


In [29]:
valeur_par_action = cours_euros.mul(
    quantites,
    axis="columns"
)

valeur_portefeuille_buy_hold = (
    valeur_par_action.sum(axis=1)
)

rendement_buy_hold = (
    valeur_portefeuille_buy_hold
    / valeur_portefeuille_buy_hold.shift(1)
) - 1

rendement_buy_hold = rendement_buy_hold.dropna()

display(
    valeur_portefeuille_buy_hold.tail().to_frame(
        name="Valeur Buy & Hold"
    ).style.format("{:,.2f} €")
)

,Valeur Buy & Hold
Date,
2026-08-17 00:00:00,"27,657.37 €"
2026-08-18 00:00:00,"27,661.05 €"
2026-08-19 00:00:00,"27,705.70 €"
2026-08-20 00:00:00,"27,194.29 €"
2026-08-21 00:00:00,"27,364.42 €"


In [30]:
valeurs_finales_actions = valeur_par_action.iloc[-1]

poids_finaux = (
    valeurs_finales_actions
    / valeur_portefeuille_buy_hold.iloc[-1]
)

comparaison_poids = pd.DataFrame({
    "Poids initial": poids,
    "Poids final": poids_finaux,
    "Valeur finale": valeurs_finales_actions
})

display(
    comparaison_poids.style.format({
        "Poids initial": "{:.2%}",
        "Poids final": "{:.2%}",
        "Valeur finale": "{:,.2f} €"
    })
)

,Poids initial,Poids final,Valeur finale
Apple,40.00%,33.65%,"9,207.56 €"
BNP Paribas,30.00%,27.22%,"7,449.64 €"
Google,30.00%,39.13%,"10,707.22 €"


In [31]:
valeur_finale_buy_hold = (
    valeur_portefeuille_buy_hold.iloc[-1]
)

performance_buy_hold = (
    valeur_finale_buy_hold / montant_initial
) - 1

rendement_annuel_buy_hold = (
    valeur_finale_buy_hold / montant_initial
) ** (252 / len(rendement_buy_hold)) - 1

volatilite_buy_hold = (
    rendement_buy_hold.std() * np.sqrt(252)
)

sharpe_buy_hold = (
    rendement_annuel_buy_hold - taux_sans_risque
) / volatilite_buy_hold

cumul_buy_hold = (
    1 + rendement_buy_hold
).cumprod()

drawdown_buy_hold = (
    cumul_buy_hold / cumul_buy_hold.cummax()
) - 1

maximum_drawdown_buy_hold = drawdown_buy_hold.min()

var_buy_hold = rendement_buy_hold.quantile(0.05)

resume_buy_hold = pd.DataFrame({
    "Montant initial": [montant_initial],
    "Valeur finale": [valeur_finale_buy_hold],
    "Performance totale": [performance_buy_hold],
    "Rendement annuel": [rendement_annuel_buy_hold],
    "Volatilité annuelle": [volatilite_buy_hold],
    "Ratio de Sharpe": [sharpe_buy_hold],
    "Maximum Drawdown": [maximum_drawdown_buy_hold],
    "VaR historique 95 %": [var_buy_hold]
}, index=["Buy & Hold"])

display(
    resume_buy_hold.style.format({
        "Montant initial": "{:,.2f} €",
        "Valeur finale": "{:,.2f} €",
        "Performance totale": "{:.2%}",
        "Rendement annuel": "{:.2%}",
        "Volatilité annuelle": "{:.2%}",
        "Ratio de Sharpe": "{:.2f}",
        "Maximum Drawdown": "{:.2%}",
        "VaR historique 95 %": "{:.2%}"
    })
)

,Montant initial,Valeur finale,Performance totale,Rendement annuel,Volatilité annuelle,Ratio de Sharpe,Maximum Drawdown,VaR historique 95 %
Buy & Hold,"10,000.00 €","27,364.42 €",173.64%,31.05%,19.78%,1.47,-25.12%,-1.83%


In [32]:
# ============================================================
# OPTIMISATION DU PORTEFEUILLE
# Simulation de 10 000 répartitions différentes
# ============================================================

# 1. Calculer les rendements annuels historiques
rendements_attendus = (
    rendements_euros.mean() * 252
)

# 2. Calculer la matrice de covariance annuelle
matrice_covariance = (
    rendements_euros.cov() * 252
)

# 3. Définir le nombre de simulations
nombre_simulations = 10_000
nombre_actions = len(rendements_euros.columns)

# 4. Créer un générateur aléatoire reproductible
generateur = np.random.default_rng(42)

# 5. Générer 10 000 répartitions
# La somme des poids de chaque portefeuille vaut 100 %
poids_simules = generateur.dirichlet(
    np.ones(nombre_actions),
    size=nombre_simulations
)

# 6. Calculer le rendement annuel de chaque portefeuille
rendements_simules = (
    poids_simules @ rendements_attendus.values
)

# 7. Calculer la volatilité annuelle de chaque portefeuille
volatilites_simulees = np.sqrt(
    np.einsum(
        "ij,jk,ik->i",
        poids_simules,
        matrice_covariance.values,
        poids_simules
    )
)

# 8. Calculer le ratio de Sharpe
sharpes_simules = (
    rendements_simules - taux_sans_risque
) / volatilites_simulees

# 9. Créer le tableau des simulations
simulations = pd.DataFrame({
    "Rendement annuel": rendements_simules,
    "Volatilité annuelle": volatilites_simulees,
    "Ratio de Sharpe": sharpes_simules
})

# 10. Ajouter les poids de chaque entreprise
for position, entreprise in enumerate(
    rendements_euros.columns
):
    simulations[f"Poids {entreprise}"] = (
        poids_simules[:, position]
    )

# 11. Identifier le portefeuille avec le meilleur Sharpe
indice_max_sharpe = (
    simulations["Ratio de Sharpe"].idxmax()
)

portefeuille_max_sharpe = (
    simulations.loc[indice_max_sharpe]
)

# 12. Identifier le portefeuille le moins volatil
indice_min_volatilite = (
    simulations["Volatilité annuelle"].idxmin()
)

portefeuille_min_volatilite = (
    simulations.loc[indice_min_volatilite]
)

# 13. Réunir les deux résultats
resultats_optimisation = pd.DataFrame({
    "Maximum Sharpe": portefeuille_max_sharpe,
    "Minimum volatilité": portefeuille_min_volatilite
}).T

# 14. Définir le format d’affichage
colonnes_pourcentage = [
    "Rendement annuel",
    "Volatilité annuelle",
    "Poids Apple",
    "Poids BNP Paribas",
    "Poids Google"
]

formats = {
    colonne: "{:.2%}"
    for colonne in colonnes_pourcentage
}

formats["Ratio de Sharpe"] = "{:.2f}"

# 15. Afficher les résultats
display(
    resultats_optimisation.style.format(formats)
)

,Rendement annuel,Volatilité annuelle,Ratio de Sharpe,Poids Apple,Poids BNP Paribas,Poids Google
Maximum Sharpe,31.69%,19.19%,1.55,21.09%,40.85%,38.06%
Minimum volatilité,29.32%,18.41%,1.48,36.12%,44.61%,19.27%


In [38]:
# ============================================================
# CRÉER ET LANCER L'APPLICATION DEPUIS JUPYTER
# ============================================================

from pathlib import Path
from IPython.display import display, HTML
import subprocess
import socket
import sys
import time


# ------------------------------------------------------------
# CODE COMPLET DE L'APPLICATION
# ------------------------------------------------------------

code_application = r'''
import warnings
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
import yfinance as yf


warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="Portfolio Risk Analyzer",
    page_icon="📈",
    layout="wide"
)


# ------------------------------------------------------------
# STYLE
# ------------------------------------------------------------

st.markdown(
    """
    <style>
        .stApp {
            background-color: #071426;
            color: white;
        }

        [data-testid="stSidebar"] {
            background-color: #0c1d34;
        }

        [data-testid="stMetric"] {
            background-color: #102641;
            border: 1px solid #1e416a;
            padding: 16px;
            border-radius: 12px;
        }

        [data-testid="stMetricLabel"] {
            color: #b9cee6;
        }

        [data-testid="stMetricValue"] {
            color: white;
        }

        h1, h2, h3 {
            color: white;
        }

        .information {
            background-color: #102641;
            border-left: 4px solid #20c7b7;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
        }
    </style>
    """,
    unsafe_allow_html=True
)


# ------------------------------------------------------------
# ENTREPRISES DISPONIBLES
# ------------------------------------------------------------

entreprises = {
    "Google": {"ticker": "GOOGL", "devise": "USD"},
    "Apple": {"ticker": "AAPL", "devise": "USD"},
    "Microsoft": {"ticker": "MSFT", "devise": "USD"},
    "Amazon": {"ticker": "AMZN", "devise": "USD"},
    "Nvidia": {"ticker": "NVDA", "devise": "USD"},
    "Meta": {"ticker": "META", "devise": "USD"},
    "Tesla": {"ticker": "TSLA", "devise": "USD"},
    "Netflix": {"ticker": "NFLX", "devise": "USD"},
    "HSBC": {"ticker": "HSBC", "devise": "USD"},
    "BNP Paribas": {"ticker": "BNP.PA", "devise": "EUR"},
    "Société Générale": {"ticker": "GLE.PA", "devise": "EUR"},
    "Crédit Agricole": {"ticker": "ACA.PA", "devise": "EUR"},
    "LVMH": {"ticker": "MC.PA", "devise": "EUR"},
    "Airbus": {"ticker": "AIR.PA", "devise": "EUR"},
    "TotalEnergies": {"ticker": "TTE.PA", "devise": "EUR"},
    "Sanofi": {"ticker": "SAN.PA", "devise": "EUR"}
}


# ------------------------------------------------------------
# FONCTIONS
# ------------------------------------------------------------

def extraire_cloture(donnees, symboles):

    if donnees.empty:
        return pd.DataFrame()

    if isinstance(donnees.columns, pd.MultiIndex):
        cloture = donnees["Close"].copy()
    else:
        cloture = donnees[["Close"]].copy()

    if isinstance(cloture, pd.Series):
        cloture = cloture.to_frame(name=symboles[0])

    if len(symboles) == 1:
        cloture.columns = [symboles[0]]

    return cloture


@st.cache_data(ttl=3600, show_spinner=False)
def telecharger_cours(symboles, debut, fin):

    donnees = yf.download(
        list(symboles),
        start=debut,
        end=fin,
        auto_adjust=True,
        progress=False,
        threads=True
    )

    return extraire_cloture(
        donnees,
        list(symboles)
    )


@st.cache_data(ttl=3600, show_spinner=False)
def telecharger_eurusd(debut, fin):

    donnees = yf.download(
        "EURUSD=X",
        start=debut,
        end=fin,
        auto_adjust=True,
        progress=False
    )

    if donnees.empty:
        return pd.Series(dtype=float)

    if isinstance(donnees.columns, pd.MultiIndex):
        change = donnees["Close"].squeeze()
    else:
        change = donnees["Close"].squeeze()

    change.name = "EUR/USD"

    return change


def style_graphique(fig):

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="#071426",
        plot_bgcolor="#071426",
        font_color="white",
        margin={"l": 20, "r": 20, "t": 60, "b": 20}
    )

    return fig


# ------------------------------------------------------------
# TITRE
# ------------------------------------------------------------

st.title("📈 Portfolio Risk Analyzer")

st.markdown(
    """
    <div class="information">
        Construisez un portefeuille d’actions, analysez sa performance,
        mesurez son risque et recherchez une allocation optimale.
    </div>
    """,
    unsafe_allow_html=True
)


# ------------------------------------------------------------
# PARAMÈTRES
# ------------------------------------------------------------

st.sidebar.header("⚙️ Paramètres")

selection = st.sidebar.multiselect(
    "Entreprises",
    list(entreprises.keys()),
    default=["Apple", "BNP Paribas", "Google"],
    max_selections=6
)

date_debut = st.sidebar.date_input(
    "Date de début",
    value=date(2023, 1, 1)
)

date_fin = st.sidebar.date_input(
    "Date de fin",
    value=date.today(),
    max_value=date.today()
)

montant_initial = st.sidebar.number_input(
    "Montant initial (€)",
    min_value=100.0,
    value=10000.0,
    step=500.0
)

taux_sans_risque = (
    st.sidebar.number_input(
        "Taux sans risque (%)",
        min_value=0.0,
        value=2.0,
        step=0.1
    ) / 100
)


if len(selection) < 2:
    st.warning("Sélectionnez au moins deux entreprises.")
    st.stop()

if date_debut >= date_fin:
    st.error("La date de début doit précéder la date de fin.")
    st.stop()


# ------------------------------------------------------------
# ALLOCATION
# ------------------------------------------------------------

st.sidebar.subheader("💶 Allocation")

poids_saisis = {}
poids_egal = 100 / len(selection)

for nom in selection:

    poids_saisis[nom] = st.sidebar.number_input(
        f"{nom} (%)",
        min_value=0.0,
        max_value=100.0,
        value=float(round(poids_egal, 2)),
        step=1.0,
        key=f"poids_{nom}"
    )

total_poids = sum(poids_saisis.values())

st.sidebar.write(f"Total saisi : **{total_poids:.2f} %**")

if total_poids <= 0:
    st.error("L’allocation ne peut pas être nulle.")
    st.stop()

if abs(total_poids - 100) > 0.1:
    st.sidebar.warning(
        "Les poids seront automatiquement ramenés à 100 %."
    )

poids = (
    pd.Series(poids_saisis, dtype=float)
    / total_poids
)


# ------------------------------------------------------------
# TÉLÉCHARGEMENT
# ------------------------------------------------------------

symboles = [
    entreprises[nom]["ticker"]
    for nom in selection
]

nom_par_symbole = {
    entreprises[nom]["ticker"]: nom
    for nom in selection
}

fin_telechargement = date_fin + timedelta(days=1)

with st.spinner("Téléchargement des données..."):

    try:
        cours = telecharger_cours(
            tuple(symboles),
            date_debut.isoformat(),
            fin_telechargement.isoformat()
        )
    except Exception as erreur:
        st.error(f"Erreur de téléchargement : {erreur}")
        st.stop()


if cours.empty:
    st.error("Aucune donnée n’a été récupérée.")
    st.stop()


cours = (
    cours
    .reindex(columns=symboles)
    .rename(columns=nom_par_symbole)
    .ffill()
    .dropna()
)


if len(cours) < 30:
    st.error("La période contient trop peu de données.")
    st.stop()


# ------------------------------------------------------------
# CONVERSION EN EUROS
# ------------------------------------------------------------

cours_euros = cours.copy()

entreprises_usd = [
    nom
    for nom in selection
    if entreprises[nom]["devise"] == "USD"
]

if entreprises_usd:

    change = telecharger_eurusd(
        date_debut.isoformat(),
        fin_telechargement.isoformat()
    )

    if change.empty:
        st.error("Impossible de récupérer le taux EUR/USD.")
        st.stop()

    change = (
        change
        .reindex(cours_euros.index)
        .ffill()
        .bfill()
    )

    for nom in entreprises_usd:
        cours_euros[nom] = cours_euros[nom] / change


# ------------------------------------------------------------
# RENDEMENTS ET PORTEFEUILLE BUY & HOLD
# ------------------------------------------------------------

rendements = (
    cours_euros / cours_euros.shift(1)
) - 1

rendements = (
    rendements
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

poids = poids.reindex(cours_euros.columns)

montants = poids * montant_initial

quantites = montants / cours_euros.iloc[0]

valeur_par_action = cours_euros.mul(
    quantites,
    axis="columns"
)

valeur_portefeuille = valeur_par_action.sum(axis=1)

rendement_portefeuille = (
    valeur_portefeuille
    / valeur_portefeuille.shift(1)
) - 1

rendement_portefeuille = rendement_portefeuille.dropna()


# ------------------------------------------------------------
# INDICATEURS
# ------------------------------------------------------------

valeur_finale = valeur_portefeuille.iloc[-1]

performance_totale = (
    valeur_finale / montant_initial
) - 1

nombre_jours = (
    valeur_portefeuille.index[-1]
    - valeur_portefeuille.index[0]
).days

nombre_annees = max(
    nombre_jours / 365.25,
    1 / 365.25
)

rendement_annuel = (
    valeur_finale / montant_initial
) ** (1 / nombre_annees) - 1

volatilite = (
    rendement_portefeuille.std()
    * np.sqrt(252)
)

ratio_sharpe = (
    rendement_annuel - taux_sans_risque
) / volatilite

cumul = (1 + rendement_portefeuille).cumprod()

drawdown = cumul / cumul.cummax() - 1

maximum_drawdown = drawdown.min()

var_95 = rendement_portefeuille.quantile(0.05)


# ------------------------------------------------------------
# ONGLETS
# ------------------------------------------------------------

tab1, tab2, tab3 = st.tabs([
    "📊 Vue générale",
    "🛡️ Risque",
    "⚙️ Optimisation"
])


# ------------------------------------------------------------
# VUE GÉNÉRALE
# ------------------------------------------------------------

with tab1:

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Valeur finale",
        f"{valeur_finale:,.2f} €",
        f"{performance_totale:.2%}"
    )

    col2.metric(
        "Volatilité",
        f"{volatilite:.2%}"
    )

    col3.metric(
        "Ratio de Sharpe",
        f"{ratio_sharpe:.2f}"
    )

    col4.metric(
        "Value at Risk",
        f"{var_95:.2%}"
    )

    col5, col6 = st.columns(2)

    col5.metric(
        "Rendement annuel",
        f"{rendement_annuel:.2%}"
    )

    col6.metric(
        "Maximum Drawdown",
        f"{maximum_drawdown:.2%}"
    )

    fig_valeur = px.line(
        x=valeur_portefeuille.index,
        y=valeur_portefeuille.values,
        title="Évolution du portefeuille",
        labels={
            "x": "Date",
            "y": "Valeur en euros"
        }
    )

    fig_valeur.update_traces(
        line_color="#20c7b7",
        line_width=3
    )

    st.plotly_chart(
        style_graphique(fig_valeur),
        use_container_width=True
    )

    performance_base100 = (
        cours_euros / cours_euros.iloc[0]
    ) * 100

    fig_performance = px.line(
        performance_base100,
        x=performance_base100.index,
        y=performance_base100.columns,
        title="Performance comparée – Base 100",
        labels={
            "value": "Performance",
            "variable": "Entreprise",
            "Date": "Date"
        }
    )

    st.plotly_chart(
        style_graphique(fig_performance),
        use_container_width=True
    )

    col_gauche, col_droite = st.columns(2)

    with col_gauche:

        repartition = pd.DataFrame({
            "Entreprise": poids.index,
            "Poids": poids.values
        })

        fig_pie = px.pie(
            repartition,
            names="Entreprise",
            values="Poids",
            title="Répartition initiale",
            hole=0.55
        )

        st.plotly_chart(
            style_graphique(fig_pie),
            use_container_width=True
        )

    with col_droite:

        valeurs_finales = valeur_par_action.iloc[-1]

        poids_finaux = (
            valeurs_finales / valeurs_finales.sum()
        )

        tableau_allocation = pd.DataFrame({
            "Entreprise": poids.index,
            "Poids initial": poids.values,
            "Poids final": poids_finaux.values,
            "Valeur finale": valeurs_finales.values
        })

        st.subheader("Évolution de l’allocation")

        st.dataframe(
            tableau_allocation.style.format({
                "Poids initial": "{:.2%}",
                "Poids final": "{:.2%}",
                "Valeur finale": "{:,.2f} €"
            }),
            use_container_width=True,
            hide_index=True
        )


# ------------------------------------------------------------
# RISQUE
# ------------------------------------------------------------

with tab2:

    correlations = rendements.corr()

    fig_corr = px.imshow(
        correlations,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title="Corrélations"
    )

    st.plotly_chart(
        style_graphique(fig_corr),
        use_container_width=True
    )

    fig_drawdown = px.area(
        x=drawdown.index,
        y=drawdown.values,
        title="Pertes depuis les précédents sommets",
        labels={
            "x": "Date",
            "y": "Drawdown"
        }
    )

    fig_drawdown.update_yaxes(tickformat=".0%")
    fig_drawdown.update_traces(
        line_color="#ff6b5f"
    )

    st.plotly_chart(
        style_graphique(fig_drawdown),
        use_container_width=True
    )

    fig_distribution = px.histogram(
        rendement_portefeuille,
        nbins=60,
        title="Distribution des rendements quotidiens"
    )

    fig_distribution.update_xaxes(
        tickformat=".1%"
    )

    st.plotly_chart(
        style_graphique(fig_distribution),
        use_container_width=True
    )


# ------------------------------------------------------------
# OPTIMISATION
# ------------------------------------------------------------

with tab3:

    rendement_attendu = rendements.mean() * 252
    covariance = rendements.cov() * 252

    nombre_simulations = 10000
    nombre_actions = len(rendements.columns)

    generateur = np.random.default_rng(42)

    poids_simules = generateur.dirichlet(
        np.ones(nombre_actions),
        size=nombre_simulations
    )

    rendements_simules = (
        poids_simules @ rendement_attendu.values
    )

    volatilites_simulees = np.sqrt(
        np.einsum(
            "ij,jk,ik->i",
            poids_simules,
            covariance.values,
            poids_simules
        )
    )

    sharpes_simules = (
        rendements_simules - taux_sans_risque
    ) / volatilites_simulees

    simulations = pd.DataFrame({
        "Rendement annuel": rendements_simules,
        "Volatilité annuelle": volatilites_simulees,
        "Ratio de Sharpe": sharpes_simules
    })

    for position, nom in enumerate(rendements.columns):
        simulations[f"Poids {nom}"] = (
            poids_simules[:, position]
        )

    meilleur_sharpe = simulations.loc[
        simulations["Ratio de Sharpe"].idxmax()
    ]

    minimum_volatilite = simulations.loc[
        simulations["Volatilité annuelle"].idxmin()
    ]

    fig_optimisation = px.scatter(
        simulations,
        x="Volatilité annuelle",
        y="Rendement annuel",
        color="Ratio de Sharpe",
        color_continuous_scale="Viridis",
        title="Simulation de 10 000 portefeuilles"
    )

    fig_optimisation.add_trace(
        go.Scatter(
            x=[meilleur_sharpe["Volatilité annuelle"]],
            y=[meilleur_sharpe["Rendement annuel"]],
            mode="markers",
            name="Maximum Sharpe",
            marker={
                "color": "red",
                "size": 18,
                "symbol": "star"
            }
        )
    )

    fig_optimisation.add_trace(
        go.Scatter(
            x=[minimum_volatilite["Volatilité annuelle"]],
            y=[minimum_volatilite["Rendement annuel"]],
            mode="markers",
            name="Minimum volatilité",
            marker={
                "color": "orange",
                "size": 16,
                "symbol": "diamond"
            }
        )
    )

    fig_optimisation.update_xaxes(tickformat=".1%")
    fig_optimisation.update_yaxes(tickformat=".1%")

    st.plotly_chart(
        style_graphique(fig_optimisation),
        use_container_width=True
    )

    resultats = pd.DataFrame({
        "Maximum Sharpe": meilleur_sharpe,
        "Minimum volatilité": minimum_volatilite
    }).T

    formats = {
        colonne: "{:.2%}"
        for colonne in resultats.columns
        if colonne != "Ratio de Sharpe"
    }

    formats["Ratio de Sharpe"] = "{:.2f}"

    st.dataframe(
        resultats.style.format(formats),
        use_container_width=True
    )


st.caption(
    "Projet pédagogique. Les performances passées ne garantissent "
    "pas les performances futures. Aucune recommandation financière."
)
'''


# ------------------------------------------------------------
# ÉCRIRE AUTOMATIQUEMENT LE FICHIER APP.PY
# ------------------------------------------------------------

fichier_application = Path("app.py")

fichier_application.write_text(
    code_application,
    encoding="utf-8"
)

print(
    f"Application écrite dans : "
    f"{fichier_application.resolve()}"
)


# ------------------------------------------------------------
# ARRÊTER L'ANCIENNE APPLICATION ÉVENTUELLE
# ------------------------------------------------------------

if "processus_streamlit" in globals():
    if processus_streamlit.poll() is None:
        processus_streamlit.terminate()
        time.sleep(1)


# ------------------------------------------------------------
# TROUVER AUTOMATIQUEMENT UN PORT DISPONIBLE
# ------------------------------------------------------------

socket_test = socket.socket()
socket_test.bind(("", 0))

port = socket_test.getsockname()[1]

socket_test.close()


# ------------------------------------------------------------
# LANCER STREAMLIT
# ------------------------------------------------------------

processus_streamlit = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(fichier_application),
        "--server.port",
        str(port),
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)


# ------------------------------------------------------------
# VÉRIFIER LE LANCEMENT
# ------------------------------------------------------------

if processus_streamlit.poll() is not None:

    erreur = processus_streamlit.stdout.read()

    raise RuntimeError(
        "L’application n’a pas pu démarrer :\n\n"
        + erreur
    )


# ------------------------------------------------------------
# AFFICHER LE BOUTON
# ------------------------------------------------------------

adresse = f"http://localhost:{port}"

display(
    HTML(
        f"""
        <div style="
            background:#102641;
            color:white;
            padding:22px;
            border-radius:12px;
            font-family:Arial;
        ">
            <h2 style="margin-top:0;">
                📈 Portfolio Risk Analyzer
            </h2>

            <p>
                L’application fonctionne maintenant.
            </p>

            <a
                href="{adresse}"
                target="_blank"
                style="
                    display:inline-block;
                    padding:13px 22px;
                    background:#20c7b7;
                    color:#071426;
                    text-decoration:none;
                    border-radius:8px;
                    font-weight:bold;
                "
            >
                Ouvrir l’application
            </a>

            <p style="margin-top:15px; margin-bottom:0;">
                {adresse}
            </p>
        </div>
        """
    )
)

Application écrite dans : /Users/nenegallely/portfolio-risk-analyzer/app.py
